In [4]:
import torch
import torch.nn as nn
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import ClippedAdam, Adam
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

import time

import torch.nn.functional as F
from pyro.nn import PyroModule, PyroSample
from sklearn.model_selection import train_test_split






In [2]:
# Set random seed for reproducibility
pyro.set_rng_seed(42)

In [6]:
# load the data
test_data_path = "small_test_data.csv"
test_label_path = "small_test_labels.csv"
test_data = pd.read_csv(test_data_path).values.astype(np.float32)
test_label = pd.read_csv(test_label_path).values

val_data_path = "small_val_data.csv"
val_label_path = "small_val_labels.csv"
val_data = pd.read_csv(val_data_path).values.astype(np.float32)
val_label = pd.read_csv(val_label_path).values

# Presentation of data

In [8]:
def normalize_data(data):
    # Z-score normalization per gene
    X_train_mean = data.mean(axis=0)
    X_train_std = data.std(axis=0) + 1e-6  # avoid div-by-zero
    X_norm = (data - X_train_mean) / X_train_std
    X_clipped = np.clip(X_norm, -15, 15)
    return X_clipped, X_train_mean, X_train_std


X_norm, X_train_mean, X_train_std = normalize_data(test_data)
X_val_norm, _, _ = normalize_data(val_data)

In [18]:
# ---------- Data Loading & Preprocessing ----------
X_train = torch.tensor(X_norm, dtype=torch.float32)
X_test = torch.tensor(X_val_norm, dtype=torch.float32)
y_train = torch.tensor(test_label, dtype=torch.float32)
y_test = torch.tensor(val_label, dtype=torch.float32)

# ---------- Pyro VAE Components ----------
input_dim = X_train.shape[1]
hidden_dim = 50
latent_dim = 20

class Encoder(PyroModule):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc21 = nn.Linear(hidden_dim, latent_dim)  # Mean
        self.fc22 = nn.Linear(hidden_dim, latent_dim)  # Log-variance

    def forward(self, x):
        h = F.relu(self.fc1(x))
        return self.fc21(h), self.fc22(h)

class Decoder(PyroModule):
    def __init__(self):
        super().__init__()
        self.fc3 = nn.Linear(latent_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, input_dim)

    def forward(self, z):
        h = F.relu(self.fc3(z))
        return self.fc4(h)  # No activation (linear output)

encoder = Encoder()
decoder = Decoder()

# ---------- Pyro Model & Guide ----------
def model(x):
    pyro.module("decoder", decoder)
    with pyro.plate("data", x.shape[0]):
        z = pyro.sample("latent", dist.Normal(torch.zeros(x.shape[0], latent_dim), torch.ones(x.shape[0], latent_dim)).to_event(1))
        loc = decoder(z)
        pyro.sample("obs", dist.Normal(loc, 1.0).to_event(1), obs=x)

def guide(x):
    pyro.module("encoder", encoder)
    with pyro.plate("data", x.shape[0]):
        z_loc, z_logvar = encoder(x)
        z_scale = torch.exp(0.5 * z_logvar)
        pyro.sample("latent", dist.Normal(z_loc, z_scale).to_event(1))

# ---------- SVI Training ----------
optimizer = Adam({"lr": 1e-3})
svi = SVI(model, guide, optimizer, loss=Trace_ELBO())

num_epochs = 3000
for epoch in range(num_epochs):
    loss = svi.step(X_train)
    print(f"Epoch {epoch+1}, Loss: {loss / len(X_train):.4f}")

# ---------- Latent Feature Extraction ----------
def extract_latents(x):
    encoder.eval()
    with torch.no_grad():
        z_loc, _ = encoder(x)
    return z_loc

Z_train = extract_latents(X_train)
Z_test = extract_latents(X_test)

# ---------- Classification (using latent space) ----------
classifier = nn.Sequential(
    nn.Linear(latent_dim, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

clf_loss_fn = nn.CrossEntropyLoss()
clf_optimizer = torch.optim.Adam(classifier.parameters(), lr=1e-3)

# Train classifier
classifier.train()
for epoch in range(3000):
    clf_optimizer.zero_grad()
    logits = classifier(Z_train)  # shape: [batch_size, 3]
    loss = clf_loss_fn(logits, y_train.squeeze().long())  # Flatten y_train to 1D
    loss.backward()
    clf_optimizer.step()
    if epoch % 10 == 0:
        print(f"Classifier Epoch {epoch+1}, Loss: {loss.item():.4f}")

# ---------- Evaluation ----------
# Evaluation
classifier.eval()
with torch.no_grad():
    logits_test = classifier(Z_test)
    y_pred_labels = torch.argmax(logits_test, dim=1).numpy()

print(confusion_matrix(y_test, y_pred_labels))
print(classification_report(y_test, y_pred_labels, target_names=["AML","ALL","Healthy"]))



Epoch 1, Loss: 581.4767
Epoch 2, Loss: 583.0640
Epoch 3, Loss: 581.4526
Epoch 4, Loss: 581.4237
Epoch 5, Loss: 581.4159
Epoch 6, Loss: 581.9852
Epoch 7, Loss: 581.8796
Epoch 8, Loss: 581.3527
Epoch 9, Loss: 582.3783
Epoch 10, Loss: 581.0647
Epoch 11, Loss: 581.2171
Epoch 12, Loss: 581.4193
Epoch 13, Loss: 582.9598
Epoch 14, Loss: 581.6156
Epoch 15, Loss: 581.5200
Epoch 16, Loss: 581.9848
Epoch 17, Loss: 581.2575
Epoch 18, Loss: 582.3663
Epoch 19, Loss: 582.4283
Epoch 20, Loss: 582.7833
Epoch 21, Loss: 582.1893
Epoch 22, Loss: 582.1142
Epoch 23, Loss: 582.8661
Epoch 24, Loss: 581.7057
Epoch 25, Loss: 581.8794
Epoch 26, Loss: 581.8504
Epoch 27, Loss: 582.0192
Epoch 28, Loss: 581.4341
Epoch 29, Loss: 581.3053
Epoch 30, Loss: 582.1130
Epoch 31, Loss: 582.4532
Epoch 32, Loss: 582.9457
Epoch 33, Loss: 582.2565
Epoch 34, Loss: 581.7941
Epoch 35, Loss: 581.8192
Epoch 36, Loss: 582.7803
Epoch 37, Loss: 581.8848
Epoch 38, Loss: 582.3479
Epoch 39, Loss: 582.0814
Epoch 40, Loss: 582.1908
Epoch 41,